# Camada Gold

A camada Gold representa a etapa analítica da Arquitetura Medalhão.

Nesta fase, serão utilizadas as bases tratadas e validadas na camada Silver para integrar informações demográficas do IBGE com dados de contribuintes da Previdência Social.

O objetivo é construir conjuntos de dados voltados às análises do projeto, permitindo observar a evolução da estrutura etária da população, indicadores de envelhecimento e a relação entre população e contribuintes previdenciários.

As transformações realizadas nesta etapa serão orientadas pelas perguntas e indicadores que se pretende analisar, preservando as bases da camada Silver como fonte dos dados tratados.

In [20]:
import pandas as pd
from pathlib import Path

pasta_silver = Path("dados/silver")
pasta_gold = Path("dados/gold")

## Leitura das bases tratadas da camada Silver

Com as cinco bases tratadas e validadas na camada Silver, inicia-se a construção da camada Gold.

Nesta etapa, serão carregadas as bases demográficas do IBGE e as bases previdenciárias do AEPS que servirão de origem para a construção dos conjuntos analíticos.

As bases possuem diferentes períodos e níveis de detalhamento. Por esse motivo, elas serão inicialmente mantidas separadas e integradas posteriormente de acordo com os objetivos de cada análise.

In [21]:
df_grupos_etarios = pd.read_csv(
    pasta_silver / "ibge_grupos_etarios_tratado.csv"
)

df_indicadores = pd.read_csv(
    pasta_silver / "ibge_indicadores_tratado.csv"
)

df_contribuintes_idade = pd.read_csv(
    pasta_silver / "aeps_contribuintes_tratado.csv"
)

df_historico_contribuintes = pd.read_csv(
    pasta_silver / "aeps_historico_contribuintes_tratado.csv"
)

df_historico_beneficios = pd.read_csv(
    pasta_silver / "aeps_historico_beneficios_tratado.csv"
)

In [22]:
print("IBGE - Grupos etários:", df_grupos_etarios.shape)
print("IBGE - Indicadores:", df_indicadores.shape)
print("AEPS - Contribuintes por idade:", df_contribuintes_idade.shape)
print("AEPS - Histórico de contribuintes:", df_historico_contribuintes.shape)
print("AEPS - Histórico de benefícios:", df_historico_beneficios.shape)

IBGE - Grupos etários: (2343, 15)
IBGE - Indicadores: (2343, 15)
AEPS - Contribuintes por idade: (42, 6)
AEPS - Histórico de contribuintes: (21, 5)
AEPS - Histórico de benefícios: (19, 5)


## Construção da base demográfica analítica

As duas bases do IBGE apresentam informações complementares sobre a evolução demográfica brasileira.

A base de grupos etários contém a distribuição da população por grandes grupos de idade, enquanto a base de indicadores reúne medidas como fecundidade, expectativa de vida, razão de dependência e índice de envelhecimento.

Como ambas abrangem o período de 2000 a 2070 e possuem informações para as mesmas localidades, será inicialmente verificada a compatibilidade das chaves utilizadas para identificar cada observação antes da integração das duas bases.

In [23]:
print("Chaves duplicadas - Grupos etários:")
print(
    df_grupos_etarios.duplicated(
        subset=["ano", "codigo", "sigla", "local"]
    ).sum()
)

print("\nChaves duplicadas - Indicadores:")
print(
    df_indicadores.duplicated(
        subset=["ano", "codigo", "sigla", "local"]
    ).sum()
)

Chaves duplicadas - Grupos etários:
0

Chaves duplicadas - Indicadores:
0


## Verificação da correspondência entre as bases demográficas

Após confirmar a unicidade das chaves nas duas bases do IBGE, será verificado se os registros de ano e localidade correspondem integralmente entre as tabelas.

Essa etapa garante que a integração seja realizada apenas após confirmar que cada observação da base de grupos etários possui uma observação correspondente na base de indicadores demográficos.

In [24]:
chaves_demograficas = [
    "ano",
    "codigo",
    "sigla",
    "local"
]

verificacao_chaves = df_grupos_etarios[
    chaves_demograficas
].merge(
    df_indicadores[chaves_demograficas],
    on=chaves_demograficas,
    how="outer",
    indicator=True
)

verificacao_chaves["_merge"].value_counts()

_merge
both          2343
left_only        0
right_only       0
Name: count, dtype: int64

## Integração das bases demográficas

A verificação confirmou que as 2.343 combinações de ano e localidade possuem correspondência integral entre as duas bases do IBGE.

Com essa compatibilidade confirmada, as informações de grupos etários e indicadores demográficos serão integradas em uma única base analítica.

Como a variável `populacao_total` está presente nas duas fontes, será mantida apenas uma ocorrência dessa informação, evitando duplicidade de variáveis na base resultante.

In [25]:
colunas_indicadores = [
    coluna for coluna in df_indicadores.columns
    if coluna not in chaves_demograficas + ["populacao_total"]
]

df_demografia_gold = df_grupos_etarios.merge(
    df_indicadores[
        chaves_demograficas + colunas_indicadores
    ],
    on=chaves_demograficas,
    how="inner",
    validate="one_to_one"
)

df_demografia_gold.shape

(2343, 25)

## Verificação da estrutura da base demográfica integrada

Após a integração das duas bases do IBGE, será verificada a estrutura do conjunto analítico resultante.

Essa etapa permite confirmar as variáveis disponíveis na base Gold e verificar se as informações de grupos etários e indicadores demográficos foram incorporadas corretamente antes da criação de novos indicadores e do armazenamento da base.

In [26]:
df_demografia_gold.info()

<class 'pandas.DataFrame'>
RangeIndex: 2343 entries, 0 to 2342
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ano                       2343 non-null   int64  
 1   codigo                    2343 non-null   int64  
 2   sigla                     2343 non-null   str    
 3   local                     2343 non-null   str    
 4   populacao_total           2343 non-null   int64  
 5   populacao_0_14            2343 non-null   int64  
 6   populacao_15_64           2343 non-null   int64  
 7   populacao_60_mais         2343 non-null   int64  
 8   populacao_65_mais         2343 non-null   int64  
 9   populacao_80_mais         2343 non-null   int64  
 10  proporcao_0_14            2343 non-null   float64
 11  proporcao_15_64           2343 non-null   float64
 12  proporcao_60_mais         2343 non-null   float64
 13  proporcao_65_mais         2343 non-null   float64
 14  proporcao_80_mais  

## Validação da população total entre as fontes

A variável `populacao_total` está presente nas duas bases demográficas do IBGE. Durante a integração, foi mantida apenas a variável proveniente da base de grupos etários para evitar duplicidade de informações.

Antes de prosseguir com a construção dos indicadores analíticos, será verificado se os valores de população total são equivalentes nas duas fontes para todas as combinações de ano e localidade.

In [27]:
validacao_populacao = df_grupos_etarios[
    chaves_demograficas + ["populacao_total"]
].merge(
    df_indicadores[
        chaves_demograficas + ["populacao_total"]
    ],
    on=chaves_demograficas,
    how="inner",
    suffixes=("_grupos", "_indicadores"),
    validate="one_to_one"
)

diferencas_populacao = (
    validacao_populacao["populacao_total_grupos"]
    != validacao_populacao["populacao_total_indicadores"]
).sum()

print(
    "Registros com diferença na população total:",
    diferencas_populacao
)

Registros com diferença na população total: 0


## Seleção da série demográfica do Brasil

A base demográfica integrada contém informações para o Brasil, regiões e Unidades da Federação. Entretanto, as séries históricas de contribuintes e benefícios do AEPS utilizadas no projeto apresentam abrangência nacional.

Para possibilitar posteriormente a análise conjunta das informações demográficas e previdenciárias, será criada uma série específica para o Brasil.

A base demográfica completa será preservada na camada Gold para análises territoriais, enquanto a série nacional será utilizada como referência para a integração com os dados históricos do AEPS.

In [28]:
df_demografia_brasil = (
    df_demografia_gold[
        df_demografia_gold["local"] == "Brasil"
    ]
    .copy()
    .reset_index(drop=True)
)

print("Dimensão:", df_demografia_brasil.shape)
print(
    "Período:",
    df_demografia_brasil["ano"].min(),
    "a",
    df_demografia_brasil["ano"].max()
)

df_demografia_brasil.head()

Dimensão: (71, 25)
Período: 2000 a 2070


,ano,codigo,sigla,local,populacao_total,populacao_0_14,populacao_15_64,populacao_60_mais,populacao_65_mais,populacao_80_mais,...,taxa_crescimento,expectativa_vida,expectativa_vida_60,taxa_fecundidade,razao_dependencia_jovens,razao_dependencia_idosos,razao_dependencia_total,indice_envelhecimento,idade_media,idade_mediana
0,2000,0,BR,Brasil,174695935,52259915,111908697,15229921,10527323,2032255,...,NaN,71.102185,20.109002,2.315552,48.747147,14.206208,62.953355,29.142644,28.314517,25.286941
1,2001,0,BR,Brasil,177003743,51985452,114176772,15648261,10841519,2097640,...,1.321043,71.502797,20.255832,2.149242,47.531716,14.307632,61.839348,30.101231,28.572942,25.595223
2,2002,0,BR,Brasil,179228254,51668727,116386875,16075850,11172652,2173094,...,1.256759,71.824406,20.361073,2.066614,46.346450,14.419914,60.766364,31.113308,28.845196,25.923454
3,2003,0,BR,Brasil,181377654,51329911,118532883,16518858,11514860,2256815,...,1.199253,72.103568,20.440327,2.016345,45.213085,14.550357,59.763442,32.181739,29.128255,26.278205
4,2004,0,BR,Brasil,183469593,50980818,120624088,16991330,11864687,2348541,...,1.153361,72.587444,20.629628,1.969960,44.140213,14.711434,58.851646,33.328869,29.420245,26.650273


## Verificação da compatibilidade temporal das séries históricas

A integração entre os dados demográficos e previdenciários exige a identificação do período em que as diferentes séries possuem informações simultaneamente disponíveis.

A série demográfica nacional do IBGE abrange o período de 2000 a 2070, enquanto a série histórica de contribuintes do AEPS compreende os anos de 2003 a 2023 e a série histórica de benefícios abrange 2005 a 2023.

Nesta etapa será identificado o intervalo temporal comum às três bases antes da construção da série histórica integrada.

In [29]:
print(
    "Demografia:",
    df_demografia_brasil["ano"].min(),
    "a",
    df_demografia_brasil["ano"].max()
)

print(
    "Contribuintes:",
    df_historico_contribuintes["ano"].min(),
    "a",
    df_historico_contribuintes["ano"].max()
)

print(
    "Benefícios:",
    df_historico_beneficios["ano"].min(),
    "a",
    df_historico_beneficios["ano"].max()
)

inicio_periodo_comum = max(
    df_demografia_brasil["ano"].min(),
    df_historico_contribuintes["ano"].min(),
    df_historico_beneficios["ano"].min()
)

fim_periodo_comum = min(
    df_demografia_brasil["ano"].max(),
    df_historico_contribuintes["ano"].max(),
    df_historico_beneficios["ano"].max()
)

print(
    "\nPeríodo comum:",
    inicio_periodo_comum,
    "a",
    fim_periodo_comum
)

Demografia: 2000 a 2070
Contribuintes: 2003 a 2023
Benefícios: 2005 a 2023

Período comum: 2005 a 2023


## Construção da série histórica integrada

Com o período comum identificado entre 2005 e 2023, os dados demográficos nacionais serão integrados às séries históricas de contribuintes e benefícios do AEPS.

Essa integração permitirá reunir, para cada ano, informações sobre a estrutura demográfica brasileira, a quantidade de contribuintes pessoas físicas e a quantidade de benefícios emitidos.

A construção dessa base estabelece o conjunto histórico central para as análises conjuntas entre a evolução demográfica e previdenciária.

In [30]:
df_historico_gold = (
    df_demografia_brasil
    .merge(
        df_historico_contribuintes,
        on="ano",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        df_historico_beneficios,
        on="ano",
        how="inner",
        validate="one_to_one"
    )
)

print("Dimensão:", df_historico_gold.shape)

print(
    "Período:",
    df_historico_gold["ano"].min(),
    "a",
    df_historico_gold["ano"].max()
)

Dimensão: (19, 33)
Período: 2005 a 2023


## Verificação da estrutura da série histórica integrada

Após a integração das três fontes, será verificada a estrutura da série histórica resultante.

Essa etapa permite confirmar que as variáveis demográficas, de contribuintes e de benefícios foram incorporadas corretamente antes da construção dos indicadores analíticos.

A inspeção também servirá de base para definir quais relações entre as variáveis são relevantes para o objetivo do estudo.

In [31]:
df_historico_gold.info()

<class 'pandas.DataFrame'>
RangeIndex: 19 entries, 0 to 18
Data columns (total 33 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ano                         19 non-null     int64  
 1   codigo                      19 non-null     int64  
 2   sigla                       19 non-null     str    
 3   local                       19 non-null     str    
 4   populacao_total             19 non-null     int64  
 5   populacao_0_14              19 non-null     int64  
 6   populacao_15_64             19 non-null     int64  
 7   populacao_60_mais           19 non-null     int64  
 8   populacao_65_mais           19 non-null     int64  
 9   populacao_80_mais           19 non-null     int64  
 10  proporcao_0_14              19 non-null     float64
 11  proporcao_15_64             19 non-null     float64
 12  proporcao_60_mais           19 non-null     float64
 13  proporcao_65_mais           19 non-null     floa

## Construção da relação entre contribuintes e benefícios

Com a série histórica integrada, torna-se possível construir indicadores que relacionem a evolução da base contributiva às informações de benefícios emitidos pelo sistema previdenciário.

O primeiro indicador será a relação entre a quantidade de contribuintes pessoas físicas e o total de benefícios emitidos em cada ano.

Esse indicador representa quantos contribuintes são observados para cada benefício emitido na série histórica. Sua interpretação deve ser realizada como uma relação entre as duas bases do AEPS, não como uma medida direta de equilíbrio financeiro ou de sustentabilidade da Previdência Social.

Também não deve ser interpretado como quantidade de trabalhadores por aposentado, pois a variável de benefícios contempla diferentes espécies de benefícios e não representa necessariamente indivíduos únicos.

In [32]:
df_historico_gold["contribuintes_por_beneficio"] = (
    df_historico_gold["total_contribuintes"]
    / df_historico_gold["total_beneficios"]
)

df_historico_gold[
    [
        "ano",
        "total_contribuintes",
        "total_beneficios",
        "contribuintes_por_beneficio"
    ]
].round(2)

,ano,total_contribuintes,total_beneficios,contribuintes_por_beneficio
0,2005,45035035,23951320,1.88
1,2006,46676737,24593390,1.90
2,2007,49936338,25170283,1.98
3,2008,53964928,26095625,2.07
4,2009,55877835,27048356,2.07
5,2010,60197924,28141263,2.14
6,2011,64292255,29051423,2.21
7,2012,67246063,30057265,2.24
8,2013,69669481,31199043,2.23
9,2014,71493806,32152518,2.22


## Relação entre contribuintes e aposentadorias

Além da relação com o total de benefícios emitidos, será analisada especificamente a quantidade de contribuintes em relação às aposentadorias.

Esse indicador permite acompanhar como a base de contribuintes se relaciona historicamente com uma das principais categorias de benefícios previdenciários e diretamente relacionada ao processo de envelhecimento populacional.

Assim como o indicador anterior, essa relação possui caráter descritivo e não representa, isoladamente, uma medida de equilíbrio financeiro ou atuarial do sistema previdenciário.

In [37]:
df_historico_gold["contribuintes_por_aposentadoria"] = (
    df_historico_gold["total_contribuintes"]
    / df_historico_gold["aposentadorias"]
)

df_historico_gold[
    [
        "ano",
        "total_contribuintes",
        "aposentadorias",
        "contribuintes_por_aposentadoria"
    ]
].round(2)

,ano,total_contribuintes,aposentadorias,contribuintes_por_aposentadoria
0,2005,45035035,13053959,3.45
1,2006,46676737,13446661,3.47
2,2007,49936338,13878747,3.60
3,2008,53964928,14453455,3.73
4,2009,55877835,15076295,3.71
5,2010,60197924,15606264,3.86
6,2011,64292255,16139303,3.98
7,2012,67246063,16725927,4.02
8,2013,69669481,17351730,4.02
9,2014,71493806,17940405,3.99


## Relação entre envelhecimento populacional e indicadores previdenciários

Após a construção dos indicadores previdenciários, será observada sua evolução em conjunto com o processo de envelhecimento da população brasileira.

Para essa análise será utilizado o `indice_envelhecimento`, disponibilizado na base de indicadores demográficos do IBGE, juntamente com as relações de contribuintes por benefício e contribuintes por aposentadoria calculadas a partir das séries históricas do AEPS.

A comparação conjunta dessas variáveis permitirá observar se as transformações na estrutura etária da população ocorreram simultaneamente a mudanças nas relações entre contribuintes e benefícios ao longo do período analisado.

Essa análise possui caráter descritivo e não estabelece, isoladamente, relação de causalidade entre envelhecimento populacional e comportamento do sistema previdenciário.

In [34]:
df_historico_gold[
    [
        "ano",
        "indice_envelhecimento",
        "contribuintes_por_beneficio",
        "contribuintes_por_aposentadoria"
    ]
].round(2)

,ano,indice_envelhecimento,contribuintes_por_beneficio,contribuintes_por_aposentadoria
0,2005,34.58,1.88,0.29
1,2006,35.97,1.90,0.29
2,2007,37.53,1.98,0.28
3,2008,39.23,2.07,0.27
4,2009,41.05,2.07,0.27
5,2010,43.05,2.14,0.26
6,2011,45.19,2.21,0.25
7,2012,47.43,2.24,0.25
8,2013,49.81,2.23,0.25
9,2014,52.28,2.22,0.25


## Relação entre contribuintes e população de 15 a 64 anos

Para complementar a análise histórica, será calculada a relação entre a quantidade de contribuintes pessoas físicas registrada pelo AEPS e a população brasileira de 15 a 64 anos apresentada pelo IBGE.

O indicador será expresso em percentual e permitirá observar a evolução da base contributiva em relação ao tamanho da população desse grupo etário ao longo do período analisado.

Essa relação não deve ser interpretada como taxa de emprego, formalização ou participação no mercado de trabalho, pois as duas bases possuem conceitos e metodologias próprios. O indicador será utilizado exclusivamente como uma medida descritiva para comparação entre as séries disponíveis no projeto.

In [35]:
df_historico_gold["contribuintes_populacao_15_64"] = (
    df_historico_gold["total_contribuintes"]
    / df_historico_gold["populacao_15_64"]
    * 100
)

df_historico_gold[
    [
        "ano",
        "populacao_15_64",
        "total_contribuintes",
        "contribuintes_populacao_15_64"
    ]
].round(2)

,ano,populacao_15_64,total_contribuintes,contribuintes_populacao_15_64
0,2005,122678705,45035035,36.71
1,2006,124716555,46676737,37.43
2,2007,126742367,49936338,39.40
3,2008,128741904,53964928,41.92
4,2009,130694593,55877835,42.75
5,2010,132415855,60197924,45.46
6,2011,133940245,64292255,48.00
7,2012,135492473,67246063,49.63
8,2013,137041821,69669481,50.84
9,2014,138534751,71493806,51.61


## Construção do cenário demográfico de referência

Após a análise histórica do período de 2005 a 2023, será observada a evolução demográfica projetada para as próximas décadas.

Como as séries históricas do AEPS utilizadas no projeto terminam em 2023, esse ano será utilizado como ponto de referência para comparação com as projeções demográficas de 2030, 2040, 2050, 2060 e 2070.

Nesta etapa não serão projetadas quantidades futuras de contribuintes, benefícios ou aposentadorias. A análise futura será restrita às projeções demográficas disponibilizadas pelo IBGE, evitando extrapolações previdenciárias sem informações suficientes para sustentá-las.

In [36]:
anos_referencia = [
    2023,
    2030,
    2040,
    2050,
    2060,
    2070
]

df_cenario_demografico = (
    df_demografia_brasil[
        df_demografia_brasil["ano"].isin(anos_referencia)
    ]
    [
        [
            "ano",
            "populacao_total",
            "populacao_15_64",
            "populacao_60_mais",
            "populacao_65_mais",
            "proporcao_15_64",
            "proporcao_60_mais",
            "proporcao_65_mais",
            "taxa_fecundidade",
            "razao_dependencia_idosos",
            "indice_envelhecimento",
            "idade_mediana"
        ]
    ]
    .reset_index(drop=True)
)

df_cenario_demografico.round(2)

,ano,populacao_total,populacao_15_64,populacao_60_mais,populacao_65_mais,proporcao_15_64,proporcao_60_mais,proporcao_65_mais,taxa_fecundidade,razao_dependencia_idosos,indice_envelhecimento,idade_mediana
0,2023,211695158,146375852,32981491,22792811,0.69,0.16,0.11,1.57,24.22,77.56,34.81
1,2030,216973093,148758847,41243737,29729196,0.69,0.19,0.14,1.47,30.05,107.17,37.85
2,2040,220386440,148048781,53633215,39575688,0.67,0.24,0.18,1.44,40.03,163.71,42.04
3,2050,218369418,137950440,65549776,50675679,0.63,0.30,0.23,1.45,53.26,220.39,45.69
4,2060,211069036,126142315,73032189,58162074,0.60,0.35,0.28,1.47,65.63,272.87,48.57
5,2070,199228708,113603532,75292150,61816833,0.57,0.38,0.31,1.50,75.20,316.24,51.17


## Variação dos principais indicadores demográficos

Para dimensionar as transformações demográficas projetadas, serão comparados os valores de 2023 e 2070.

A comparação permitirá quantificar as mudanças na população em idade de 15 a 64 anos, na população idosa e nos principais indicadores relacionados ao envelhecimento populacional.

Essas variações serão utilizadas posteriormente como apoio à interpretação dos possíveis desafios demográficos associados ao sistema previdenciário, sem realizar projeções de contribuintes, benefícios ou equilíbrio financeiro.

In [38]:
dados_2023 = df_demografia_brasil[
    df_demografia_brasil["ano"] == 2023
].iloc[0]

dados_2070 = df_demografia_brasil[
    df_demografia_brasil["ano"] == 2070
].iloc[0]

variacao_15_64 = (
    (dados_2070["populacao_15_64"] - dados_2023["populacao_15_64"])
    / dados_2023["populacao_15_64"]
    * 100
)

variacao_60_mais = (
    (dados_2070["populacao_60_mais"] - dados_2023["populacao_60_mais"])
    / dados_2023["populacao_60_mais"]
    * 100
)

variacao_65_mais = (
    (dados_2070["populacao_65_mais"] - dados_2023["populacao_65_mais"])
    / dados_2023["populacao_65_mais"]
    * 100
)

print(f"População de 15 a 64 anos: {variacao_15_64:.2f}%")
print(f"População de 60 anos ou mais: {variacao_60_mais:.2f}%")
print(f"População de 65 anos ou mais: {variacao_65_mais:.2f}%")

População de 15 a 64 anos: -22.39%
População de 60 anos ou mais: 128.29%
População de 65 anos ou mais: 171.21%


## Validação dos indicadores da série histórica

Após a construção dos indicadores analíticos, será verificada a existência de valores ausentes ou infinitos resultantes dos cálculos realizados.

Essa validação garante que as relações entre contribuintes, benefícios, aposentadorias e população de 15 a 64 anos estejam disponíveis para todos os anos da série histórica integrada antes do armazenamento da base na camada Gold.

In [39]:
indicadores_historicos = [
    "contribuintes_por_beneficio",
    "contribuintes_por_aposentadoria",
    "contribuintes_populacao_15_64"
]

print(
    "Valores ausentes:",
    df_historico_gold[indicadores_historicos]
    .isnull()
    .sum()
)

print(
    "\nValores infinitos:",
    (
        df_historico_gold[indicadores_historicos]
        .isin([float("inf"), float("-inf")])
        .sum()
    )
)

Valores ausentes: contribuintes_por_beneficio        0
contribuintes_por_aposentadoria    0
contribuintes_populacao_15_64      0
dtype: int64

Valores infinitos: contribuintes_por_beneficio        0
contribuintes_por_aposentadoria    0
contribuintes_populacao_15_64      0
dtype: int64
